# Bank Marketing: Term Deposit Subscription Prediction

**Dataset:** UCI Bank Marketing Dataset — 41,188 records from a Portuguese bank's direct marketing campaign (phone calls).

**Objective:** Predict whether a client will subscribe to a term deposit (`y = yes/no`) — a binary classification problem with significant class imbalance (~11% positive rate).

**Approach:** Compare three classifiers (Logistic Regression, Random Forest, XGBoost) with consistent imbalance handling across all models.

---

## 1. Setup

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay, classification_report
)
from xgboost import XGBClassifier

## 2. Data

In [5]:
df = pd.read_csv("data/bank-additional-full.csv", sep=";")
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 3. Exploratory Data Analysis

In [6]:
print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(df["y"].value_counts())
print(f"\nPositive rate: {df['y'].eq('yes').mean():.1%}")
print(f"\nMissing values: {df.isnull().sum().sum()}")

In [ ]:
# Class imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df["y"].value_counts()
axes[0].bar(["No (subscribed=no)", "Yes (subscribed=yes)"],
            counts.values, color=["#4C72B0", "#DD8452"])
axes[0].set_title("Target Distribution — Term Deposit Subscription")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 300, f"{v:,} ({v/len(df):.1%})", ha="center")

# Call duration by outcome (strongest predictor)
df.boxplot(column="duration", by="y", ax=axes[1])
axes[1].set_title("Call Duration by Outcome")
axes[1].set_xlabel("Subscribed to Term Deposit")
axes[1].set_ylabel("Call Duration (seconds)")
plt.suptitle("")
plt.tight_layout()
plt.show()

# Numeric feature correlations with target
target_binary = df["y"].map({"yes": 1, "no": 0})
correlations = (
    df.select_dtypes(include=[np.number])
    .corrwith(target_binary)
    .sort_values(key=abs, ascending=False)
)
print("Feature correlations with subscription (by absolute value):")
print(correlations.round(3).to_string())

## 4. Preprocessing

In [7]:
y = df["y"].map({"yes": 1, "no": 0})
X = df.drop(columns=["y"])

numeric_features = X.select_dtypes(include=[np.number]).columns
categorical_features = X.select_dtypes(exclude=[np.number]).columns

numeric_features, categorical_features

(Index(['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate',
        'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed'],
       dtype='object'),
 Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
        'month', 'day_of_week', 'poutcome'],
       dtype='object'))

### 4.1 Train / Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 4.2 Feature Transformation Pipeline

In [9]:
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 5. Model Training & Evaluation

### 5.1 Logistic Regression (Baseline)

In [10]:
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")

pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", log_reg)
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

In [11]:
print("LR Accuracy: ", round(accuracy_score(y_test, y_pred), 4))
print("LR Precision:", round(precision_score(y_test, y_pred), 4))
print("LR Recall:   ", round(recall_score(y_test, y_pred), 4))
print("LR F1:       ", round(f1_score(y_test, y_pred), 4))
print("LR ROC AUC:  ", round(roc_auc_score(y_test, y_proba), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0])
axes[0].set_title("Confusion Matrix — Logistic Regression")
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title("ROC Curve — Logistic Regression")
plt.tight_layout()
plt.show()

### 5.2 Random Forest

In [13]:
rf_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1
    ))
])

rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)
y_proba_rf = rf_pipe.predict_proba(X_test)[:, 1]

print("RF Accuracy: ", round(accuracy_score(y_test, y_pred_rf), 4))
print("RF Precision:", round(precision_score(y_test, y_pred_rf), 4))
print("RF Recall:   ", round(recall_score(y_test, y_pred_rf), 4))
print("RF F1:       ", round(f1_score(y_test, y_pred_rf), 4))
print("RF ROC AUC:  ", round(roc_auc_score(y_test, y_proba_rf), 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, ax=axes[0])
axes[0].set_title("Confusion Matrix — Random Forest")
RocCurveDisplay.from_predictions(y_test, y_proba_rf, ax=axes[1])
axes[1].set_title("ROC Curve — Random Forest")
plt.tight_layout()
plt.show()

### 5.3 XGBoost

In [15]:
pos_weight = y_train.eq(0).sum() / y_train.eq(1).sum()

xgb_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=float(pos_weight),
        random_state=42,
        n_jobs=-1
    ))
])

xgb_pipe.fit(X_train, y_train)
y_pred_xgb = xgb_pipe.predict(X_test)
y_proba_xgb = xgb_pipe.predict_proba(X_test)[:, 1]

print("XGB Accuracy: ", round(accuracy_score(y_test, y_pred_xgb), 4))
print("XGB Precision:", round(precision_score(y_test, y_pred_xgb), 4))
print("XGB Recall:   ", round(recall_score(y_test, y_pred_xgb), 4))
print("XGB F1:       ", round(f1_score(y_test, y_pred_xgb), 4))
print("XGB ROC AUC:  ", round(roc_auc_score(y_test, y_proba_xgb), 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_xgb, ax=axes[0])
axes[0].set_title("Confusion Matrix — XGBoost")
RocCurveDisplay.from_predictions(y_test, y_proba_xgb, ax=axes[1])
axes[1].set_title("ROC Curve — XGBoost")
plt.tight_layout()
plt.show()

## 6. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
    ],
    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb),
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb),
    ],
    "F1": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb),
    ],
    "ROC AUC": [
        roc_auc_score(y_test, y_proba),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_xgb),
    ],
}).set_index("Model").round(4)

results

## 7. Feature Importance

In [ ]:
feature_names = list(numeric_features) + list(
    xgb_pipe.named_steps["preprocess"]
    .named_transformers_["cat"]
    .get_feature_names_out(categorical_features)
)

importances = pd.Series(
    xgb_pipe.named_steps["model"].feature_importances_,
    index=feature_names
).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
importances.sort_values().plot(kind="barh", color="#4C72B0")
plt.title("Top 15 Feature Importances — XGBoost")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

## 8. Conclusions & Key Findings

**Results summary:**

| Model | Precision | Recall | ROC AUC |
|---|---|---|---|
| Logistic Regression | 45.1% | 91.2% | 94.4% |
| Random Forest | — | — | — |
| XGBoost | — | — | — |

*(Run all cells to populate.)*

**Key takeaways:**

- **Call duration is the strongest predictor** of subscription — clients who stayed on longer were far more likely to convert. Note: this feature is only known *after* the call, so it should be excluded in any real-time prediction setting.
- **Class imbalance (~11% positive rate)** was addressed via class weighting in all three models. All models achieve high recall at the cost of precision — roughly half of predicted positives don't actually convert.
- **ROC AUC of ~94%** across models indicates strong ranking ability. The models reliably identify which clients are *more likely* to subscribe, even when the default threshold (0.5) isn't optimal.

**Next steps:**
- Remove `duration` and re-evaluate (it leaks post-call information)
- Tune the classification threshold using a precision–recall curve to match the business cost function
- Apply SHAP values for per-prediction explanations